# MiniGPT — Treinamento no Google Colab

Notebook completo pra treinar o MiniGPT usando a GPU gratuita do Colab.

**Como usar:**
1. Vá em `Ambiente de execução > Alterar o tipo de ambiente de execução` e selecione **T4 GPU**
2. Execute cada célula em ordem (Shift+Enter)
3. No final, baixe os arquivos de output

In [ ]:
#@title 1. Verificar GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('⚠️ GPU não detectada! Vá em Ambiente de execução > Alterar tipo > T4 GPU')

In [ ]:
#@title 2. Criar arquivos do projeto
%%bash
mkdir -p minigpt/data minigpt/output

cat > minigpt/config.py << 'PYEOF'
"""
config.py — Central de hiperparâmetros do MiniGPT

Aqui definimos TODOS os hiperparâmetros do modelo em um só lugar.

DICAS DIDÁTICAS:
- d_model: dimensão interna do modelo. GPT-2 small usa 768, nós usamos 128
  pq queremos rodar em CPU rapidamente.
- n_heads: número de cabeças de atenção. A dimensão por cabeça é
  d_model // n_heads (aqui: 128/4 = 32). Precisa ser divisível!
- n_layers: quantos blocos Transformer empilhamos. Mais camadas =
  mais capacidade, mas mais lento pra treinar.
- context_len: tamanho da janela de contexto. Quantos tokens o
  modelo consegue "olhar pra trás" de uma vez.
- dropout: probabilidade de "desligar" neurônios aleatoriamente
  durante o treino. Previne overfitting. 0.1 = 10%.
"""


from dataclasses import dataclass


@dataclass
class GPTConfig:
    # --- Dimensões do modelo ---
    d_model: int = 128
    n_heads: int = 4
    n_layers: int = 4

    # --- Janela de contexto ---
    context_len: int = 128

    # --- Regularização ---
    dropout: float = 0.1

    # --- Treinamento ---
    batch_size: int = 64
    learning_rate: float = 3e-4
    weight_decay: float = 0.1
    max_epochs: int = 30

    # --- Otimizador ---
    beta1: float = 0.9
    beta2: float = 0.95

    # --- Gradient clipping ---
    max_grad_norm: float = 1.0

    # --- Taxa de aprendizado com warmup + cosine decay ---
    warmup_steps: int = 100
    min_lr: float = 1e-5

    @property
    def head_dim(self) -> int:
        """Dimensão de cada cabeça de atenção."""
        assert self.d_model % self.n_heads == 0, (
            f"d_model ({self.d_model}) deve ser divisível por "
            f"n_heads ({self.n_heads})"
        )
        return self.d_model // self.n_heads
PYEOF

cat > minigpt/tokenizer.py << 'PYEOF'
"""tokenizer.py — Tokenizador char-level do MiniGPT"""

import json
from pathlib import Path

TOKENS_ESPECIAIS = ["<PAD>", "<UNK>", "<BOS>", "<EOS>"]


class CharTokenizer:
    def __init__(self):
        self.char_to_id: dict[str, int] = {}
        self.id_to_char: dict[int, str] = {}
        self.vocab_size: int = 0

    def treinar(self, texto: str) -> None:
        chars_unicos = sorted(set(texto))
        self.char_to_id = {}
        self.id_to_char = {}
        for i, token in enumerate(TOKENS_ESPECIAIS):
            self.char_to_id[token] = i
            self.id_to_char[i] = token
        for i, ch in enumerate(chars_unicos, start=len(TOKENS_ESPECIAIS)):
            self.char_to_id[ch] = i
            self.id_to_char[i] = ch
        self.vocab_size = len(self.char_to_id)

    def codificar(self, texto: str) -> list[int]:
        return [self.char_to_id.get(ch, self.char_to_id["<UNK>"]) for ch in texto]

    def decodificar(self, ids: list[int]) -> str:
        chars = []
        for id_ in ids:
            ch = self.id_to_char.get(id_, "")
            if ch in TOKENS_ESPECIAIS:
                continue
            chars.append(ch)
        return "".join(chars)

    def salvar(self, caminho: str) -> None:
        dados = {
            "char_to_id": self.char_to_id,
            "id_to_char": {str(k): v for k, v in self.id_to_char.items()},
            "vocab_size": self.vocab_size,
        }
        Path(caminho).write_text(json.dumps(dados, ensure_ascii=False, indent=2))

    @classmethod
    def carregar(cls, caminho: str) -> "CharTokenizer":
        dados = json.loads(Path(caminho).read_text())
        tok = cls()
        tok.char_to_id = dados["char_to_id"]
        tok.id_to_char = {int(k): v for k, v in dados["id_to_char"].items()}
        tok.vocab_size = dados["vocab_size"]
        return tok

    def __repr__(self) -> str:
        return f"CharTokenizer(vocab_size={self.vocab_size})"
PYEOF

cat > minigpt/model.py << 'PYEOF'
"""model.py — O modelo GPT do zero"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from config import GPTConfig

class LayerNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.gain = nn.Parameter(torch.ones(d_model))
        self.bias = nn.Parameter(torch.zeros(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)
        return self.gain * (x - mean) / torch.sqrt(var + self.eps) + self.bias

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.n_heads = config.n_heads
        self.head_dim = config.head_dim
        self.W_qkv = nn.Linear(config.d_model, 3 * config.d_model)
        self.W_out = nn.Linear(config.d_model, config.d_model)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.W_qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        attn_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mascara_causal = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        attn_scores = attn_scores.masked_fill(mascara_causal, float("-inf"))
        attn_probs = F.softmax(attn_scores, dim=-1)
        attn_probs = self.attn_dropout(attn_probs)
        out = attn_probs @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.W_out(out)
        return self.resid_dropout(out)

class FeedForward(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.d_model, 4 * config.d_model),
            nn.GELU(),
            nn.Linear(4 * config.d_model, config.d_model),
            nn.Dropout(config.dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.ln1 = LayerNorm(config.d_model)
        self.attn = MultiHeadSelfAttention(config)
        self.ln2 = LayerNorm(config.d_model)
        self.ffn = FeedForward(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class GPTModel(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.position_embedding = nn.Embedding(config.context_len, config.d_model)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.Sequential(*[TransformerBlock(config) for _ in range(config.n_layers)])
        self.ln_f = LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self._init_weights()

    def _init_weights(self) -> None:
        n_layers = self.config.n_layers
        for nome, modulo in self.named_modules():
            if isinstance(modulo, nn.Linear):
                nn.init.normal_(modulo.weight, mean=0.0, std=0.02)
                if modulo.bias is not None:
                    nn.init.zeros_(modulo.bias)
            elif isinstance(modulo, nn.Embedding):
                nn.init.normal_(modulo.weight, mean=0.0, std=0.02)
            if nome.endswith(("ffn.net.0", "W_qkv", "W_out")):
                nn.init.normal_(modulo.weight, mean=0.0, std=0.02 / math.sqrt(2 * n_layers))

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device).unsqueeze(0)
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(pos)
        x = self.drop(tok_emb + pos_emb)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def contar_parametros(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def resumo(self) -> str:
        n = self.contar_parametros()
        c = self.config
        lines = [
            f"════════════════ MiniGPT — Resumo do Modelo ════════════════",
            f"  Parâmetros:     {n:,}",
            f"  Vocabulário:    {c.vocab_size}",
            f"  d_model:        {c.d_model}",
            f"  n_heads:        {c.n_heads}",
            f"  head_dim:       {c.head_dim}",
            f"  n_layers:       {c.n_layers}",
            f"  context_len:    {c.context_len}",
            f"  dropout:        {c.dropout}",
            f"  FFN dim:        {4 * c.d_model}",
            f"════════════════════════════════════════════════════════════",
        ]
        return "\n".join(lines)
PYEOF

cat > minigpt/generate.py << 'PYEOF'
"""generate.py — Geração autoregressiva de texto"""

import torch
import torch.nn.functional as F
from model import GPTModel
from tokenizer import CharTokenizer

@torch.no_grad()
def gerar(modelo, tokenizer, prompt, max_tokens=200, temperature=0.8, top_k=40, device="cpu"):
    modelo.eval()
    tokens = tokenizer.codificar(prompt)
    idx = torch.tensor([tokens], dtype=torch.long, device=device)
    eos_id = tokenizer.char_to_id.get("<EOS>", -1)
    for _ in range(max_tokens):
        idx_cond = idx[:, -modelo.config.context_len:]
        logits, _ = modelo(idx_cond)
        logits = logits[:, -1, :]
        logits = logits / max(temperature, 1e-8)
        if top_k is not None and top_k > 0:
            kth_vals = torch.topk(logits, min(top_k, logits.size(-1)))[0][:, -1:]
            logits = logits.masked_fill(logits < kth_vals, float("-inf"))
        probs = F.softmax(logits, dim=-1)
        proximo_token = torch.multinomial(probs, num_samples=1)
        if proximo_token.item() == eos_id:
            break
        idx = torch.cat([idx, proximo_token], dim=1)
    return tokenizer.decodificar(idx[0].tolist())

def gerar_variacoes(modelo, tokenizer, prompt, n_variacoes=3, max_tokens=200, temperature=0.8, top_k=40, device="cpu"):
    return [gerar(modelo, tokenizer, prompt, max_tokens, temperature, top_k, device) for _ in range(n_variacoes)]
PYEOF

cat > minigpt/data/__init__.py << 'PYEOF'
PYEOF

cat > minigpt/data/corpus.py << 'PYEOF'
"""data/corpus.py — Gerador de dataset sintético em português"""
from pathlib import Path

def gerar_corpus() -> str:
    sujeitos = ["o gato", "a gata", "o cachorro", "a cachorra", "o pássaro",
        "o menino", "a menina", "o professor", "a professora", "o artista"]
    verbos = ["comia", "bebia", "dormia", "corria", "pulava",
        "brincava", "estudava", "trabalhava", "cantava", "dançava",
        "falava", "ouvia", "via", "pensava", "sonhava"]
    objetos = ["na praça", "no parque", "na casa", "na escola", "no jardim",
        "na rua", "na praia", "no rio", "na montanha", "na cidade"]
    complementos = ["com alegria", "com cuidado", "com calma", "com atenção",
        "com amor", "com paciência", "com vontade", "sem pressa"]
    frases = []
    for s in sujeitos:
        for v in verbos:
            for o in objetos[:5]:
                frases.append(f"{s.capitalize()} {v} {o}.")
    for s in sujeitos[:5]:
        for v in verbos[:5]:
            for o in objetos[:3]:
                for c in complementos[:4]:
                    frases.append(f"{s.capitalize()} {v} {o} {c}.")
    historias = [
        "Era uma vez um gato que morava numa casa muito bonita. O gato gostava de dormir no jardim todas as tardes. Um dia, o gato encontrou um pássaro na árvore. O pássaro cantava uma melodia linda. O gato e o pássaro ficaram amigos para sempre.",
        "A menina estudava todas as noites na biblioteca. Ela lia muitos livros sobre ciência e arte. A menina queria ser uma grande cientista. A professora ajudava a menina com as dúvidas. A menina trabalhava com dedicação e alegria.",
        "O cachorro corria no parque todas as manhãs. Ele gostava de brincar com a bola. O cachorro era muito feliz. A dona do cachorro cuidava dele com muito amor. Juntos, eles caminhavam pela cidade.",
        "Na cidade grande, as pessoas corriam para todo lado. O trânsito era intenso e barulhento. Mas no parque, tudo era calmo e tranquilo. As crianças brincavam na praça. Os velhos sentavam no banco e observavam os pássaros.",
        "O professor explicava a lição com paciência. Os estudantes ouviam com atenção. A aula era sobre a natureza e os animais. A menina fez uma pergunta. O professor respondeu com alegria. A turma aprendeu muito naquele dia.",
        "A cozinheira preparava um jantar delicioso. Ela cozinhava com muito cuidado. O jantar tinha arroz, feijão e salada. A família comeu com alegria. Todos elogiaram a comida da cozinheira.",
        "O artista pintava um quadro muito bonito. Ele usava cores vivas e fortes. O quadro mostrava uma paisagem da montanha. As pessoas admiravam a pintura na galeria. O artista era feliz com o seu trabalho.",
        "A cientista trabalhava no laboratório. Ela estudava as estrelas e os planetas. A cientista descobriu uma nova estrela. A descoberta era muito importante. O mundo inteiro celebrava a descoberta da cientista.",
        "No jardim da casa, as flores cresciam com beleza. O sol brilhava e a chuva caía com cuidado. As borboletas voavam entre as flores. O gato dormia na grama verde. Era um dia tranquilo e bonito.",
        "O menino e a menina brincavam na praia. Eles construíam castelos de areia. O mar era azul e calmo. Os pássaros voavam sobre a água. A tarde era quente e ensolarada. As crianças estavam felizes.",
    ]
    frases.extend(historias * 5)
    frases_soltas = ["O sol brilha no céu azul.", "A chuva cai na cidade.",
        "O vento sopra na montanha.", "A lua ilumina a noite.",
        "As estrelas brilham no escuro.", "O rio corre para o mar.",
        "A floresta é verde e bonita.", "O fogo esquenta a casa.",
        "A água é importante para a vida.", "A natureza é maravilhosa.",
        "O dia amanheceu bonito.", "O pôr do sol era lindo.",
        "A noite estava estrelada.", "A manhã chegou com alegria.",
        "A tarde foi tranquila.", "O aprendizado nunca termina.",
        "A leitura abre portas.", "O conhecimento transforma vidas.",
        "A amizade é um tesouro.", "O amor é a maior força.",
        "A vida é bela e curta.", "O tempo passa depressa.",
        "A esperança nunca morre.", "O mundo é cheio de surpresas.",
        "A música alegra o coração."]
    frases.extend(frases_soltas * 10)
    return " ".join(frases)

def salvar_corpus(caminho: str = "data/corpus.txt") -> str:
    texto = gerar_corpus()
    Path(caminho).write_text(texto, encoding="utf-8")
    return texto

def carregar_corpus(caminho: str = "data/corpus.txt") -> str:
    path = Path(caminho)
    if path.exists():
        return path.read_text(encoding="utf-8")
    return salvar_corpus(caminho)
PYEOF

echo "Arquivos criados em minigpt/"

In [ ]:
#@title 3. Treinar o modelo { display-mode: "form" }

#@markdown ## Configuração (ajuste se quiser)
D_MODEL = 128        #@param {type:"integer"}
N_HEADS = 4          #@param {type:"integer"}
N_LAYERS = 4         #@param {type:"integer"}
CONTEXT_LEN = 128    #@param {type:"integer"}
BATCH_SIZE = 64       #@param {type:"integer"}
MAX_EPOCHS = 30       #@param {type:"integer"}
LEARNING_RATE = 3e-4 #@param {type:"number"}

import sys
sys.path.insert(0, 'minigpt')

import torch
from torch.utils.data import DataLoader
import json
import math
import time
from pathlib import Path

from config import GPTConfig
from model import GPTModel
from tokenizer import CharTokenizer
from data.corpus import gerar_corpus
from generate import gerar

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, tokens, context_len):
        self.tokens = tokens
        self.context_len = context_len
    def __len__(self):
        return max(0, len(self.tokens) - self.context_len)
    def __getitem__(self, idx):
        chunk = self.tokens[idx:idx + self.context_len + 1]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y

def get_lr(it, config):
    if it < config.warmup_steps:
        return config.learning_rate * (it + 1) / config.warmup_steps
    progress = min((it - config.warmup_steps) / max(1, 10000 - config.warmup_steps), 1.0)
    coeff = 0.5 * (1.0 + math.cos(math.pi * progress))
    return config.min_lr + coeff * (config.learning_rate - config.min_lr)

# Detectar GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')

# Config
config = GPTConfig(
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    context_len=CONTEXT_LEN,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    learning_rate=LEARNING_RATE,
)

# Corpus
texto = gerar_corpus()
print(f'Corpus: {len(texto):,} caracteres')

# Tokenizer
tokenizer = CharTokenizer()
tokenizer.treinar(texto)
config.vocab_size = tokenizer.vocab_size
print(f'{tokenizer}')

# Dataset
tokens = tokenizer.codificar(texto)
dataset = TextDataset(tokens, config.context_len)
dataloader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True, pin_memory=(device != 'cpu'))
print(f'Batches por época: {len(dataloader)}')

# Modelo
modelo = GPTModel(config).to(device)
print(modelo.resumo())
n_params = modelo.contar_parametros()

# Otimizador
param_decay = [p for n, p in modelo.named_parameters() if p.dim() >= 2]
param_nodecay = [p for n, p in modelo.named_parameters() if p.dim() < 2]
otimizador = torch.optim.AdamW([
    {"params": param_decay, "weight_decay": config.weight_decay},
    {"params": param_nodecay, "weight_decay": 0.0},
], lr=config.learning_rate, betas=(config.beta1, config.beta2))

# Treino
modelo.train()
step_global = 0
melhor_loss = float('inf')
total_tokens_treinados = 0
tempo_inicio = time.time()
log_epocas = []
saida_path = Path('minigpt/output')
saida_path.mkdir(exist_ok=True)

print(f'\nTreinando por {config.max_epochs} épocas em {device}...')
print('=' * 70)

for epoca in range(1, config.max_epochs + 1):
    epoca_loss = 0.0
    n_batches = 0
    t0 = time.time()

    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        lr = get_lr(step_global, config)
        for pg in otimizador.param_groups:
            pg['lr'] = lr
        logits, loss = modelo(x, y)
        otimizador.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), config.max_grad_norm)
        otimizador.step()
        epoca_loss += loss.item()
        n_batches += 1
        step_global += 1
        total_tokens_treinados += x.shape[0] * x.shape[1]

    media_loss = epoca_loss / max(n_batches, 1)
    elapsed = time.time() - t0
    perplexidade = math.exp(min(media_loss, 20))

    log_epocas.append({
        'epoca': epoca, 'loss': round(media_loss, 6), 'ppl': round(perplexidade, 2),
        'lr': lr, 'tempo_seg': round(elapsed, 1),
    })

    print(f'Época {epoca:3d}/{config.max_epochs} │ Loss: {media_loss:.4f} │ PPL: {perplexidade:.2f} │ LR: {lr:.2e} │ Tempo: {elapsed:.1f}s')

    if media_loss < melhor_loss:
        melhor_loss = media_loss
        torch.save({'modelo': modelo.state_dict(), 'config': config, 'epoch': epoca,
                     'loss': media_loss, 'total_tokens': total_tokens_treinados,
                     'tempo_total_seg': time.time() - tempo_inicio},
                    saida_path / 'melhor_modelo.pt')
        tokenizer.salvar(str(saida_path / 'tokenizer.json'))

tempo_total = time.time() - tempo_inicio
tokens_por_seg = total_tokens_treinados / max(tempo_total, 1)

torch.save({'modelo': modelo.state_dict(), 'config': config, 'epoch': config.max_epochs,
             'loss': media_loss, 'total_tokens': total_tokens_treinados,
             'tempo_total_seg': tempo_total}, saida_path / 'modelo_final.pt')

# Log
log_treino = {
    'config': {k: getattr(config, k) for k in ['d_model','n_heads','n_layers','context_len','dropout','batch_size','learning_rate','weight_decay','max_epochs','warmup_steps','min_lr','max_grad_norm','vocab_size']},
    'device': device, 'n_params': n_params,
    'total_tokens_treinados': total_tokens_treinados,
    'tempo_total_seg': round(tempo_total, 1), 'tempo_total_min': round(tempo_total/60, 1),
    'tokens_por_segundo': round(tokens_por_seg, 1),
    'melhor_loss': round(melhor_loss, 6),
    'epocas': log_epocas,
}
(saida_path / 'log_treino.json').write_text(json.dumps(log_treino, indent=2, ensure_ascii=False))

print('=' * 70)
print(f'Concluído! Loss: {melhor_loss:.4f} | Tempo: {tempo_total:.1f}s ({tempo_total/60:.1f} min) | {tokens_por_seg:,.0f} tokens/s')
print(f'Modelo salvo em: {saida_path/"melhor_modelo.pt"}')
print(f'Modelos mapeados para CPU para compatibilidade.')

In [ ]:
#@title 4. Testar o modelo treinado
modelo.eval()
modelo_cpu = modelo.to('cpu')
print("Gerando amostras:\n")
for prompt in ["O gato", "A menina", "Na cidade", "Era uma vez", "O professor", "A chuva"]:
    resultado = gerar(modelo_cpu, tokenizer, prompt, max_tokens=120, temperature=0.8, top_k=40, device='cpu')
    print(f'Prompt: "{prompt}"')
    print(f'Saída:  {resultado}\n')

In [ ]:
#@title 5. Baixar resultados
from google.colab import files

# Mapear modelo para CPU antes de salvar (compatibilidade)
modelo.to('cpu')
torch.save({'modelo': modelo.state_dict(), 'config': config, 'epoch': config.max_epochs,
             'loss': media_loss, 'total_tokens': total_tokens_treinados,
             'tempo_total_seg': tempo_total}, 'output_modelo_final.pt')

print('Baixando arquivos...')
files.download('minigpt/output/melhor_modelo.pt')
files.download('minigpt/output/tokenizer.json')
files.download('minigpt/output/log_treino.json')
files.download('output_modelo_final.pt')